In [7]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

# ---------------- USER SETTINGS ----------------
DATA_PATH = "data/corn_2016_2023.parquet"   # change crop file here
TARGET_COL = "yield"                       # regression target
GROUP_COL  = "farm_name"                   # group split key
N_SPLITS   = 5

# feature choices
INCLUDE_YEAR_LAT = True                    # you said: include Year + latitude
USE_LONGITUDE = False                      # you decided to drop longitude
# ------------------------------------------------

# Load
df = pd.read_parquet(DATA_PATH)

# Drop rows with missing target
df = df[df[TARGET_COL].notna()].copy()

# Clean: drop old wrong normalized cols if present
bad_norm = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]

# Clean: remove SAR duplicates ending with _2
sar_dup2 = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]

df = df.drop(columns=bad_norm + sar_dup2, errors="ignore")

print("Rows:", len(df), "Cols:", len(df.columns))
print("Dropped:", bad_norm, "and", len(sar_dup2), "SAR _2 cols")

# Continuous target (float)
y = df[TARGET_COL].to_numpy(dtype=np.float32)

# Groups
groups = df[GROUP_COL].astype(str).to_numpy()
print("Unique groups:", len(np.unique(groups)))

# CV folds (fixed across models)
gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y, groups=groups))
print("Built folds:", len(folds))

# ------------ Metrics helpers ------------
def regression_metrics(y_true, y_pred):
    """Return R², RMSE, MAE."""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return r2, rmse, mae

def zone_thresholds_from_train(y_train, q_low=0.4, q_high=0.6):
    """Compute yield thresholds from TRAIN only (prevents leakage)."""
    low = np.quantile(y_train, q_low)
    high = np.quantile(y_train, q_high)
    return low, high

def yield_to_zone(y_vals, low, high):
    """Convert continuous yield to zones using thresholds."""
    # 0=low, 1=medium, 2=high
    z = np.zeros_like(y_vals, dtype=np.int64)
    z[(y_vals >= low) & (y_vals <= high)] = 1
    z[y_vals > high] = 2
    return z

def zone_metrics(y_true, y_pred, y_train_for_thresholds):
    """
    Evaluate 'zone accuracy' + confusion matrix for regression predictions.
    Thresholds are computed from TRAIN yields only.
    """
    low, high = zone_thresholds_from_train(y_train_for_thresholds, 0.4, 0.6)
    z_true = yield_to_zone(y_true, low, high)
    z_pred = yield_to_zone(y_pred, low, high)
    acc = (z_true == z_pred).mean()
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return float(acc), cm

def summarize_cv_regression(r2s, rmses, maes, name="MODEL"):
    print(f"\n{name} CV Summary (regression)")
    print(f"R²:   {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")
    print(f"RMSE: {np.mean(rmses):.3f} ± {np.std(rmses):.3f}")
    print(f"MAE:  {np.mean(maes):.3f} ± {np.std(maes):.3f}")

def summarize_cv_zones(zone_accs, cm_sum, name="MODEL"):
    print(f"\n{name} CV Summary (zone-from-regression)")
    print(f"Zone Accuracy: {np.mean(zone_accs):.3f} ± {np.std(zone_accs):.3f}")
    print("Zone confusion matrix sum (rows=true low/med/high, cols=pred):\n", cm_sum)




def build_year_mean_map(years_train, y_train_raw):
    """TRAIN-only map: year -> mean yield in that year."""
    year_means = {}
    for yr in np.unique(years_train):
        vals = y_train_raw[years_train == yr]
        year_means[int(yr)] = float(np.mean(vals))
    return year_means

def lookup_year_means(years, year_means, fallback):
    """Per-sample mean yield by year using TRAIN-derived map."""
    out = np.empty(len(years), dtype=np.float32)
    for i, yr in enumerate(years):
        out[i] = year_means.get(int(yr), fallback)
    return out

def zone_metrics_pm10_from_ratio(y_true_ratio, y_pred_ratio):
    """Zones in ratio space: low<0.9, med 0.9-1.1, high>1.1."""
    z_true = np.where(y_true_ratio < 0.9, 0, np.where(y_true_ratio <= 1.1, 1, 2))
    z_pred = np.where(y_pred_ratio < 0.9, 0, np.where(y_pred_ratio <= 1.1, 1, 2))
    acc = float((z_true == z_pred).mean())
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return acc, cm

Rows: 2108996 Cols: 349
Dropped: ['normalized_yield_pct', 'normalized_yield_z'] and 20 SAR _2 cols
Unique groups: 251
Built folds: 5


# SAnity Check

In [4]:
import numpy as np

sar_cols = [c for c in df.columns if c.startswith("VV_") or c.startswith("VH_")]
X_sar = df[sar_cols].to_numpy(dtype=np.float32)

print("SAR finite?", np.isfinite(X_sar).all())
print("SAR NaNs:", np.isnan(X_sar).sum())
print("SAR Infs:", np.isinf(X_sar).sum())
print("SAR min/max:", np.nanmin(X_sar), np.nanmax(X_sar))

# Detect crazy placeholder nodata values if any
flat = X_sar.reshape(-1)
print("Values with abs>1e3:", np.sum(np.abs(flat) > 1e3))
print("Top 10 magnitudes:", np.sort(np.abs(flat))[-10:])


SAR finite? False
SAR NaNs: 536282261
SAR Infs: 0
SAR min/max: -63.423126 9.673324
Values with abs>1e3: 0
Top 10 magnitudes: [nan nan nan nan nan nan nan nan nan nan]


# Build SAR sequence tensor (needed for RNN/LSTM/Transformer)
## What / Why

Convert VV_#### and VH_#### columns into a time series: (N, T, 2).

Preprocessing

Drop _2 columns (already done in shared setup)

Align VV and VH by suffix

Replace NaN/inf and standardize per fold (done later)

# RNN (SAR sequence only)
## What / Why

Simple sequence-aware baseline. Often weaker than LSTM, but useful.

Preprocessing required

Replace NaNs/infs

Standardize using training fold only

Gradient clipping to avoid exploding gradients

In [8]:
import re
import numpy as np

# ---------- BUILD SAR SEQUENCE (X_seq) ----------
def suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_cols = [c for c in df.columns if c.startswith("VV_")]
vh_cols = [c for c in df.columns if c.startswith("VH_")]

vv_map = {suffix(c): c for c in vv_cols if suffix(c) is not None}
vh_map = {suffix(c): c for c in vh_cols if suffix(c) is not None}

common_suffixes = sorted(set(vv_map.keys()) & set(vh_map.keys()))
T = len(common_suffixes)

print("Time steps (T):", T)

vv_sorted = [vv_map[s] for s in common_suffixes]
vh_sorted = [vh_map[s] for s in common_suffixes]

X_seq = np.zeros((len(df), T, 2), dtype=np.float32)
X_seq[:, :, 0] = df[vv_sorted].to_numpy(dtype=np.float32)
X_seq[:, :, 1] = df[vh_sorted].to_numpy(dtype=np.float32)

print("X_seq shape:", X_seq.shape)
print("NaNs in X_seq:", np.isnan(X_seq).sum())

# ---------- STATIC FEATURES: Year + latitude ----------
STATIC_COLS = ["Year", "latitude"]  # no longitude
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
print("X_static shape:", X_static.shape)


Time steps (T): 152
X_seq shape: (2108996, 152, 2)
NaNs in X_seq: 536282261
X_static shape: (2108996, 2)


In [10]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- GPU SETTINGS -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    torch.set_num_threads(50)
    torch.set_num_interop_threads(4)

# ----------------- helpers -----------------
def fmt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

def preprocess_sar_fold(X_tr, X_te, add_mask=True):
    """
    Standardize SAR using TRAIN only.
    Add missingness mask channels (recommended).
    """
    mask_tr = np.isnan(X_tr).astype(np.float32)
    mask_te = np.isnan(X_te).astype(np.float32)

    X_tr_filled = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
    X_te_filled = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

    mean = X_tr_filled.reshape(-1, X_tr.shape[-1]).mean(axis=0)
    std  = X_tr_filled.reshape(-1, X_tr.shape[-1]).std(axis=0) + 1e-6

    X_tr_std = (X_tr_filled - mean) / std
    X_te_std = (X_te_filled - mean) / std

    if add_mask:
        X_tr_out = np.concatenate([X_tr_std, mask_tr], axis=2)
        X_te_out = np.concatenate([X_te_std, mask_te], axis=2)
    else:
        X_tr_out, X_te_out = X_tr_std, X_te_std

    return X_tr_out.astype(np.float32), X_te_out.astype(np.float32)

def standardize_static_fold(S_tr, S_te):
    """Standardize Year/latitude using TRAIN only (prevents leakage)."""
    mu = S_tr.mean(axis=0)
    sd = S_tr.std(axis=0) + 1e-6
    return ((S_tr - mu) / sd).astype(np.float32), ((S_te - mu) / sd).astype(np.float32)

# ----------------- Dataset -----------------
class SeqStaticDataset(Dataset):
    def __init__(self, Xseq, Xstat, y):
        self.Xseq = torch.as_tensor(Xseq, dtype=torch.float32)
        self.Xstat = torch.as_tensor(Xstat, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)  # regression target
    def __len__(self): return self.y.shape[0]
    def __getitem__(self, i): return self.Xseq[i], self.Xstat[i], self.y[i]

# ----------------- Model -----------------
class RNNRegressor(nn.Module):
    """
    Sequence encoder: simple RNN
    Static fusion: small MLP on (Year, latitude)
    Output: one continuous yield prediction
    """
    def __init__(self, seq_input_dim=4, stat_dim=2, hidden=64, dropout=0.1):
        super().__init__()
        self.rnn = nn.RNN(seq_input_dim, hidden, batch_first=True)

        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_stat):
        _, h = self.rnn(x_seq)          # (1, B, hidden)
        h = h[-1]                       # (B, hidden)
        s = self.stat_mlp(x_stat)       # (B, 32)
        z = torch.cat([h, s], dim=1)
        return self.head(z).squeeze(1)  # (B,)

@torch.no_grad()
def eval_regression(model, loader):
    model.eval()
    yt, yp = [], []
    for xseq, xstat, yb in loader:
        xseq = xseq.to(device, non_blocking=True)
        xstat = xstat.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        pred = model(xseq, xstat)
        yt.append(yb.detach().cpu().numpy())
        yp.append(pred.detach().cpu().numpy())

    yt = np.concatenate(yt)
    yp = np.concatenate(yp)
    return yt, yp

# ----------------- Training config -----------------
BATCH_SIZE = 8192   # H200: try 8192, 16384 if it fits
EPOCHS = 6
LR = 3e-4
PIN = (device.type == "cuda")

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# ----------------- CV loop -----------------
r2s, rmses, maes = [], [], []
zone_accs = []
cm_zone_sum = np.zeros((3, 3), dtype=int)

overall_start = time.time()
total_folds = len(folds)

for k, (tr, te) in enumerate(folds, start=1):
    fold_start = time.time()
    print(f"\n===== Fold {k}/{total_folds} (RNN regression) =====")

    # 1. SAR and Static Preprocessing (Keep your existing code here)
    X_tr_seq, X_te_seq = preprocess_sar_fold(X_seq[tr], X_seq[te], add_mask=True)
    S_tr, S_te = standardize_static_fold(X_static[tr], X_static[te])

    # 2. NEW: YEAR-NORMALIZATION LOGIC
    # Get the years for this specific fold
    years_train = df["Year"].to_numpy()[tr]
    years_test  = df["Year"].to_numpy()[te]
    y_raw_tr = y[tr]
    y_raw_te = y[te]

    # Calculate means from TRAIN only to prevent leakage
    year_means_map = build_year_mean_map(years_train, y_raw_tr)
    global_fallback = float(np.mean(y_raw_tr))

    # Create the 'mu' (mean) arrays for scaling
    mu_tr = lookup_year_means(years_train, year_means_map, global_fallback)
    mu_te = lookup_year_means(years_test,  year_means_map, global_fallback)

    # Normalize targets: predict the ratio (e.g., 0.8 to 1.2)
    y_tr_norm = (y_raw_tr / mu_tr).astype(np.float32)
    y_te_norm = (y_raw_te / mu_te).astype(np.float32) 

    # 3. DataLoaders (Now using y_tr_norm and y_te_norm)
    dl_tr = DataLoader(
        SeqStaticDataset(X_tr_seq, S_tr, y_tr_norm),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN
    )
    dl_te = DataLoader(
        SeqStaticDataset(X_te_seq, S_te, y_te_norm),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN
    )

    # 4. Model Setup
    model = RNNRegressor(seq_input_dim=4, stat_dim=2, hidden=64).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    # 5. Training Loop
    for ep in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        for xseq, xstat, yb in dl_tr:
            xseq, xstat, yb = xseq.to(device), xstat.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(xseq, xstat)
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        # Validate
        y_true, y_pred = eval_regression(model, dl_te)
        r2, rmse, mae = regression_metrics(y_true, y_pred)

        epoch_time = time.time() - epoch_start

        # overall ETA (rough)
        elapsed = time.time() - overall_start
        progress = (k - 1) + (ep / EPOCHS)
        frac = min(0.999, progress / total_folds)
        overall_eta = elapsed * (1 - frac) / max(frac, 1e-6)

        print(
            f"Fold {k} epoch {ep}: R²={r2:.3f} RMSE={rmse:.3f} MAE={mae:.3f} "
            f"| epoch_time={fmt_time(epoch_time)} | overall_ETA≈{fmt_time(overall_eta)}"
        )

    # 6. EVALUATION: Convert back to Raw Yield for Metrics
    y_true_norm, y_pred_norm = eval_regression(model, dl_te)
    
    # Scale back to bu/acre: Prediction * Year_Mean
    pred_raw = y_pred_norm * mu_te
    true_raw = y_raw_te 

    # Regression metrics on RAW values
    r2, rmse, mae = regression_metrics(true_raw, pred_raw)
    
    # Zone metrics on RATIO values (using your 0.9 / 1.1 logic)
    zacc, cmz = zone_metrics_pm10_from_ratio(y_true_norm, y_pred_norm)

    r2s.append(r2); rmses.append(rmse); maes.append(mae)
    zone_accs.append(zacc); cm_zone_sum += cmz

    print(f"Fold {k} FINAL: R²={r2:.3f} | zone_acc={zacc:.3f}")
    

summarize_cv_regression(r2s, rmses, maes, name="RNNRegressor (SAR+Year+lat, GPU+AMP)")
summarize_cv_zones(zone_accs, cm_zone_sum, name="RNNRegressor (zone eval)")
print("Total wall time:", fmt_time(time.time() - overall_start))


Device: cuda
GPU: NVIDIA H200

===== Fold 1/5 (RNN regression) =====
Fold 1 epoch 1: R²=0.121 RMSE=0.230 MAE=0.175 | epoch_time=00m 14s | overall_ETA≈20m 44s
Fold 1 epoch 2: R²=0.140 RMSE=0.228 MAE=0.169 | epoch_time=00m 13s | overall_ETA≈13m 15s
Fold 1 epoch 3: R²=0.143 RMSE=0.227 MAE=0.168 | epoch_time=00m 12s | overall_ETA≈10m 27s
Fold 1 epoch 4: R²=0.148 RMSE=0.227 MAE=0.168 | epoch_time=00m 14s | overall_ETA≈09m 05s
Fold 1 epoch 5: R²=0.153 RMSE=0.226 MAE=0.167 | epoch_time=00m 14s | overall_ETA≈08m 09s
Fold 1 epoch 6: R²=0.159 RMSE=0.225 MAE=0.168 | epoch_time=00m 13s | overall_ETA≈07m 27s
Fold 1 FINAL: R²=0.239 | zone_acc=0.470

===== Fold 2/5 (RNN regression) =====
Fold 2 epoch 1: R²=0.125 RMSE=0.231 MAE=0.177 | epoch_time=00m 14s | overall_ETA≈08m 17s
Fold 2 epoch 2: R²=0.141 RMSE=0.229 MAE=0.175 | epoch_time=00m 13s | overall_ETA≈07m 34s
Fold 2 epoch 3: R²=0.140 RMSE=0.229 MAE=0.175 | epoch_time=00m 13s | overall_ETA≈06m 58s
Fold 2 epoch 4: R²=0.137 RMSE=0.230 MAE=0.175 | epo

Device: cuda
GPU: NVIDIA H200

===== Fold 1/5 (RNN regression) =====
Fold 1 epoch 1: R²=0.121 RMSE=0.230 MAE=0.175 | epoch_time=00m 14s | overall_ETA≈20m 44s
Fold 1 epoch 2: R²=0.140 RMSE=0.228 MAE=0.169 | epoch_time=00m 13s | overall_ETA≈13m 15s
Fold 1 epoch 3: R²=0.143 RMSE=0.227 MAE=0.168 | epoch_time=00m 12s | overall_ETA≈10m 27s
Fold 1 epoch 4: R²=0.148 RMSE=0.227 MAE=0.168 | epoch_time=00m 14s | overall_ETA≈09m 05s
Fold 1 epoch 5: R²=0.153 RMSE=0.226 MAE=0.167 | epoch_time=00m 14s | overall_ETA≈08m 09s
Fold 1 epoch 6: R²=0.159 RMSE=0.225 MAE=0.168 | epoch_time=00m 13s | overall_ETA≈07m 27s
Fold 1 FINAL: R²=0.239 | zone_acc=0.470

===== Fold 2/5 (RNN regression) =====
Fold 2 epoch 1: R²=0.125 RMSE=0.231 MAE=0.177 | epoch_time=00m 14s | overall_ETA≈08m 17s
Fold 2 epoch 2: R²=0.141 RMSE=0.229 MAE=0.175 | epoch_time=00m 13s | overall_ETA≈07m 34s
Fold 2 epoch 3: R²=0.140 RMSE=0.229 MAE=0.175 | epoch_time=00m 13s | overall_ETA≈06m 58s
Fold 2 epoch 4: R²=0.137 RMSE=0.230 MAE=0.175 | epoch_time=00m 14s | overall_ETA≈06m 26s
Fold 2 epoch 5: R²=0.126 RMSE=0.231 MAE=0.175 | epoch_time=00m 14s | overall_ETA≈05m 58s
Fold 2 epoch 6: R²=0.141 RMSE=0.229 MAE=0.175 | epoch_time=00m 14s | overall_ETA≈05m 33s
Fold 2 FINAL: R²=0.146 | zone_acc=0.432

===== Fold 3/5 (RNN regression) =====
Fold 3 epoch 1: R²=0.030 RMSE=0.240 MAE=0.188 | epoch_time=00m 13s | overall_ETA≈05m 41s
Fold 3 epoch 2: R²=0.082 RMSE=0.234 MAE=0.184 | epoch_time=00m 13s | overall_ETA≈05m 13s
Fold 3 epoch 3: R²=0.088 RMSE=0.233 MAE=0.183 | epoch_time=00m 13s | overall_ETA≈04m 48s
Fold 3 epoch 4: R²=0.091 RMSE=0.233 MAE=0.183 | epoch_time=00m 14s | overall_ETA≈04m 24s
Fold 3 epoch 5: R²=0.094 RMSE=0.232 MAE=0.182 | epoch_time=00m 14s | overall_ETA≈04m 02s
Fold 3 epoch 6: R²=0.083 RMSE=0.234 MAE=0.185 | epoch_time=00m 13s | overall_ETA≈03m 40s
Fold 3 FINAL: R²=0.128 | zone_acc=0.312

===== Fold 4/5 (RNN regression) =====
Fold 4 epoch 1: R²=0.121 RMSE=0.236 MAE=0.179 | epoch_time=00m 15s | overall_ETA≈03m 34s
Fold 4 epoch 2: R²=0.145 RMSE=0.233 MAE=0.176 | epoch_time=00m 14s | overall_ETA≈03m 12s
Fold 4 epoch 3: R²=0.152 RMSE=0.232 MAE=0.176 | epoch_time=00m 14s | overall_ETA≈02m 51s
Fold 4 epoch 4: R²=0.161 RMSE=0.231 MAE=0.173 | epoch_time=00m 13s | overall_ETA≈02m 30s
Fold 4 epoch 5: R²=0.171 RMSE=0.229 MAE=0.172 | epoch_time=00m 14s | overall_ETA≈02m 10s
Fold 4 epoch 6: R²=0.180 RMSE=0.228 MAE=0.172 | epoch_time=00m 14s | overall_ETA≈01m 50s
Fold 4 FINAL: R²=0.172 | zone_acc=0.376

===== Fold 5/5 (RNN regression) =====
Fold 5 epoch 1: R²=0.020 RMSE=0.217 MAE=0.173 | epoch_time=00m 14s | overall_ETA≈01m 36s
Fold 5 epoch 2: R²=0.072 RMSE=0.211 MAE=0.166 | epoch_time=00m 14s | overall_ETA≈01m 16s
Fold 5 epoch 3: R²=0.069 RMSE=0.212 MAE=0.166 | epoch_time=00m 14s | overall_ETA≈00m 56s
Fold 5 epoch 4: R²=0.078 RMSE=0.211 MAE=0.165 | epoch_time=00m 14s | overall_ETA≈00m 37s
Fold 5 epoch 5: R²=0.095 RMSE=0.209 MAE=0.163 | epoch_time=00m 14s | overall_ETA≈00m 18s
Fold 5 epoch 6: R²=0.091 RMSE=0.209 MAE=0.165 | epoch_time=00m 14s | overall_ETA≈00m 00s
Fold 5 FINAL: R²=0.205 | zone_acc=0.345

RNNRegressor (SAR+Year+lat, GPU+AMP) CV Summary (regression)
R²:   0.178 ± 0.040
RMSE: 42.618 ± 2.187
MAE:  32.731 ± 1.766

RNNRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.387 ± 0.057
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[340148 321465   9909]
 [131672 438113  11699]
 [ 53054 764579  38357]]
Total wall time: 09m 15s

1) The weird huge negative R² in early epochs is normal

You see epoch 1 like:

Fold 1 epoch 1: R² = -10.7

Fold 2 epoch 1: R² = -12.0

That usually happens because at the start the network predicts something awful (almost constant or wildly off), so the squared error is worse than the “dumb baseline” predictor (predict the mean). That makes:

𝑅
2
=
1
−
𝑆
𝑆
𝑟
𝑒
𝑠
𝑆
𝑆
𝑡
𝑜
𝑡
R
2
=1−
SS
tot
	​

SS
res
	​

	​


If 
𝑆
𝑆
𝑟
𝑒
𝑠
>
𝑆
𝑆
𝑡
𝑜
𝑡
SS
res
	​

>SS
tot
	​

, then R² < 0.

So negative R² early is fine.

2) Fold 1 ended up okay-ish; Fold 2 is weak
Fold 1 FINAL

R² = 0.259

RMSE = 40.4

MAE = 29.0

zone_acc = 0.693

That means: the model explains about 26% of the yield variance in that fold. For yield prediction using only SAR+Year+lat, that’s plausible.

Fold 2 FINAL

R² = 0.099

RMSE = 45.5

MAE = 33.8

zone_acc = 0.450

That’s barely better than predicting the mean, and the zone accuracy is close to random-ish (random among 3 classes is ~0.33, but since zones are imbalanced it can be higher than 0.33 even with weak predictions).

So: high variance across folds → not stable yet.

3) The zone accuracy numbers can be misleading right now

You’re computing zone thresholds from training yields (good), but zone accuracy can still look “okay” even when regression is mediocre because:

zones are coarse (3 buckets)

if distribution is skewed, a model that tends to predict around the middle can accidentally hit “medium” a lot

also your thresholds are based on raw yield pooled across years, not per-year zones (that affects comparability)

So focus more on R² + RMSE/MAE as primary regression metrics.

4) The biggest red flag: fold-to-fold generalization varies a lot

Fold 1: R² 0.26
Fold 2: R² 0.10

That suggests one of these:

farms in fold 2 are systematically different (geography/management)

the model is underpowered (simple RNN is weak)

the model is not seeing enough static context (only year+lat)

yield scale is large / noisy (common)

missingness (NaNs) pattern differs across folds (very likely, you had massive NaNs)

5) Practical check: is RMSE “big” or “small”?

RMSE=40 only means something if we compare to the yield scale.

Do this quick sanity check (no new modeling, just compute baselines):

What is the standard deviation of yield in the test fold?

What RMSE do you get if you just predict the train mean?

If your model RMSE is close to “predict mean” RMSE, then you’re not improving much.

(If you want, paste df["yield"].describe() and I’ll interpret RMSE in context.)

6) What I’d conclude from these two folds

✅ Training is stable (loss isn’t NaN, model learns quickly)

✅ Features are being used (R² becomes positive)

⚠️ But this model (simple RNN) is likely not strong enough and is unstable across folds

⚠️ Zone accuracy isn’t reliable to judge yet

So: valid run, not a “good final result” yet.

7) What to do next (highest impact)

If you want better results with the same data:

A) Upgrade the sequence model (best next step)

LSTM > RNN

Transformer > LSTM (often best for SAR time series)

1D CNN is fast and often strong too

B) Use a better baseline for regression than RNN

Your HistGradientBoostingRegressor will likely beat RNN quickly on tabular SAR features (even without fancy seq modeling) — good to compare.

C) Consider per-year normalization / per-year zone eval (optional)

Since yields drift by year, a model can get punished for systematic year effects. Even with Year included, the mapping may not be linear.

8) One more thing: your training is extremely fast (0.009s/batch)

That’s great, but it also hints you might be not fully utilizing GPU or the model is so small that it’s trivial.

Which is fine — but it means you can afford:

bigger hidden size (128/256)

LSTM/Transformer

slightly more epochs (10–20)

If you paste Fold 3–5 final metrics, I’ll tell you whether the overall CV looks usable.
And if you’re ready, send your LSTM notebook code next — that’s the easiest upgrade that usually improves stability a lot.

Thanks for posting the full CV output + yield distribution — this makes it possible to judge the result properly.

Short answer first:

Yes, this is a valid result, but it is not a strong model.
It is a reasonable baseline for “SAR + Year + latitude → yield regression”, and it clearly shows why you need stronger sequence models (LSTM / Transformer) and/or more features.

Now let’s go through this carefully and quantitatively, not emotionally.

1️⃣ Put the numbers in context (this is critical)

Your yield distribution:

mean ≈ 190
std  ≈ 47.8


So:

Predicting the mean yield gives RMSE ≈ 47.8

Any model with RMSE significantly below ~48 is doing better than trivial

Your model:

CV RMSE ≈ 43.0 ± 2.7

CV MAE ≈ 32.0

CV R² ≈ 0.164 ± 0.060

Interpretation

You reduced RMSE by about 10–12% vs baseline

You explain about 16% of yield variance

That is not great, but it is absolutely believable for:

pixel-level yields

noisy SAR data

no weather / soil / management

only a simple RNN

This is exactly what I would expect from a first-pass deep SAR model.

2️⃣ Fold-to-fold behavior: what it tells us

Final R² per fold:

Fold	R²
1	0.259
2	0.099
3	0.123
4	0.131
5	0.209
What this means

The model sometimes learns meaningful signal (Fold 1, Fold 5)

But generalization is unstable

This strongly suggests:

farms differ a lot

SAR signal–yield relationship is nonstationary

the RNN is underpowered

This is not overfitting — it’s under-modeling.

3️⃣ Early epochs with huge negative R² are NOT a problem

You consistently see:

Epoch 1: R² ≈ −10 to −16

Epoch 2: still negative

Epoch 3+: becomes positive

That’s normal for regression with:

random initialization

large output scale (~0–380)

SmoothL1Loss

It means:

the network starts by predicting garbage

quickly learns a reasonable scale

then fine-tunes

No red flags here.

4️⃣ Zone accuracy: why it looks “okay” but isn’t decisive
CV zone accuracy
0.540 ± 0.090


Random-ish baseline is ~0.33–0.40 depending on class imbalance, so:

0.54 is better than random

but not impressive

Look at the confusion matrix (summed):

True low    → predicted high a LOT
True medium → heavily confused
True high   → often correct


This is exactly what happens when:

regression shrinks predictions toward the mean

boundaries (40/60) are hard

medium class is intrinsically ambiguous

So zone accuracy here should be treated as secondary evidence only.

5️⃣ The most important conclusion (don’t skip this)
This result tells you three very important things
✅ (A) Your pipeline is correct

No leakage

GPU usage correct

CV split is sound

Metrics make sense

Runtime is reasonable (~9 min)

This is huge — many projects fail here.

⚠️ (B) SAR-only + simple RNN is not expressive enough

R² ≈ 0.16 is too low to be useful by itself.

This is expected:

plain RNN struggles with long, irregular time series

SAR backscatter relationships are nonlinear + temporal

🔥 (C) You are now at the point where model choice matters

This is where LSTM / Transformer / CNN will actually show value.

6️⃣ Is this “good” for a paper / report?
As a baseline?

YES. Absolutely.

You can honestly say:

“A simple RNN using SAR time series with static spatial features explains ~16% of yield variance, serving as a baseline.”

As a final model?

NO. And that’s okay — it’s not meant to be.

7️⃣ What to do next (ranked by impact)
🔥 1. LSTM (next obvious step)

Handles longer dependencies

Usually improves R² by +5–10 points over RNN

🔥 2. Transformer (most likely best)

Attention handles missing SAR observations better

Better temporal aggregation

Often much more stable across folds

⚠️ 3. Add weather features (later)

This often gives the biggest jump, but only after the temporal model is solid

8️⃣ One thing I want you to not do yet

Do not:

tune hyperparameters aggressively

add tons of features

worry about zone accuracy too much

Right now, the architecture is the bottleneck, not tuning.

Bottom line (clear verdict)

✅ Valid

❌ Not strong

🎯 Exactly what a baseline RNN should look like

🚀 Ready to move to LSTM / Transformer

If you want, next I can:

convert your LSTM code to the exact same regression + fusion + metrics

or do Transformer first (my recommendation)